В этом блокноте мы исследуем важные концепции в машинном обучении, связанные с обучением нейронных сетей: недообучение и переобучение. Мы продемонстрируем, как эти проблемы проявляются, и изучим методы борьбы с переобучением, такие как Dropout и L2-регуляризация. Мы будем использовать датасет Fashion MNIST для наглядной демонстрации этих концепций.  Особое внимание уделим анализу кривых обучения и оценке качества моделей на тестовых данных.

### Импортируем данные и необходимые библиотеки

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

Будем использовать лишь фрагмент из всего набора данных, чтобы сократить временные затраты на обучение моделей.

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train[:5000]
X_test = X_test[:1000]
y_train = y_train[:5000]
y_test = y_test[:1000]

Данные — значения интенсивностей пикселей и лежат в пределах $[0,255]$. "Распрямим" и масштабируем, чтобы получить значения из отрезка $[0,1]$.

In [ ]:
X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0

Задача — многоклассовая классификация, поэтому произведем `one-hot` кодирование откликов.

In [ ]:
y_train = np.eye(10)[y_train]

### Базовая модель

Построим самую простую модель: каждому классу ставим в соответствие один нейрон.
* Активация — `softmax`
* Оптимизатор — `adam`
* Лосс — `categorical_crossentropy`
* Метрика — `accuracy`
* Число эпох обучения — `10`
* `batch_size=256`
* `validation_split=0.2`


In [ ]:
base_model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(10, activation='softmax')
])

base_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

history = base_model.fit(X_train, y_train,
                                   epochs=10,
                                   batch_size=256,
                                   validation_split=0.2,
                                   verbose=1)

График лосса для `train` и `validation`

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()

Что можно сказать по графику?

Оценим модель на тестовых данных

In [ ]:
predictions_logits = base_model.predict(X_test)
y_pred = np.argmax(predictions_logits, axis=1)
accuracy_score(y_test, y_pred)

Увеличим число эпох обучения до 30

In [ ]:
base_model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(10, activation='softmax')
])

base_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

history = base_model.fit(X_train, y_train,
                                   epochs=30,
                                   batch_size=256,
                                   validation_split=0.2,
                                   verbose=0)

Графики лосса

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()

Что можно сказать по графику?

Оценим модель на тестовых данных

In [ ]:
predictions_logits = base_model.predict(X_test)
y_pred = np.argmax(predictions_logits, axis=1)
accuracy_score(y_test, y_pred)

### Усложняем модель

Очевидно, предыдущая модель слишком простая, хотя на деле не так плоха. Изменим архитектуру:

* Скрытые слои со следующим числом нейронов: `(1024, 512, 256)`. Активация — `relu`
* Установим `verbose=1` в методе `.fit()`, чтобы остлеживать прогресс
* Остальные параметры без изменений

In [ ]:
overfit_model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(1024, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

overfit_model.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

history = overfit_model.fit(X_train, y_train,
                          epochs=30,
                          batch_size=256,
                          validation_split=0.2,
                          verbose=1)

График лоссов

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Overfitting Demonstration')
plt.legend()
plt.show()

Что можно сказать, глядя на графики лоссов? А если еще увеличить число эпох?

Оценка на тесте

In [ ]:
predictions_logits = overfit_model.predict(X_test)
y_pred = np.argmax(predictions_logits, axis=1)
accuracy_score(y_test, y_pred)

### Регуляризация

Для борьбы с переобучением часто используют Dropout и регуляризацию. Dropout случайным образом "выключает" часть нейронов во время обучения, что заставляет сети "перераспределять" обязанности нейронов и не слишком полагаться на конкретные нейроны. Регуляризация добавляет штраф к функции потерь за большие значения весов, что препятствует сети слишком сильно подстраиваться под тренировочные данные.

In [ ]:
regularized_model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(512, activation='relu', kernel_regularizer='l2'),
    layers.Dropout(0.2),
    layers.Dense(256, activation='relu', kernel_regularizer='l2'),
    layers.Dropout(0.1),
    layers.Dense(10, activation='softmax')
])

regularized_model.compile(optimizer='adam',
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

history = regularized_model.fit(X_train, y_train,
                          epochs=30,
                          batch_size=256,
                          validation_split=0.2,
                          verbose=1)

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Good Fitting Demonstration')
plt.legend()
plt.show()

In [ ]:
predictions_logits = regularized_model.predict(X_test)
y_pred = np.argmax(predictions_logits, axis=1)
accuracy_score(y_test, y_pred)

### Больше данных

Объем данных существенно влияет на обучение и качество сети. Возьмем весь набор данных и обучим на нем самую простую и самую сложную из рассмотренных моделей.

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0
y_train = np.eye(10)[y_train]

#### Маленькая модель

In [ ]:
small_model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(10, activation='softmax')
])

small_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

history = small_model.fit(X_train, y_train,
                                   epochs=30,
                                   batch_size=256,
                                   validation_split=0.2,
                                   verbose=0)

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()

In [ ]:
predictions_logits = small_model.predict(X_test)
y_pred = np.argmax(predictions_logits, axis=1)
accuracy_score(y_test, y_pred)

Видно, что дальнейшее увеличение данных при данной архитектуре дает некоторый прирост метрики, но, похоже мы выжали из этой модели максимум.

#### Большая модель

Возьмем сразу не очень большое число эпох, например, 10

In [ ]:
big_model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(1024, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

big_model.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

history = big_model.fit(X_train, y_train,
                          epochs=10,
                          batch_size=256,
                          validation_split=0.2,
                          verbose=1)

In [ ]:
predictions_logits = big_model.predict(X_test)
y_pred = np.argmax(predictions_logits, axis=1)
accuracy_score(y_test, y_pred)

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()